# Credit Card Fraud Detection System 💳🚨

**Objective:** Build a robust machine learning classification pipeline to accurately identify fraudulent credit card transactions while minimizing false positives.

### Resume Highlights Covered:
- **Built a fraud detection system** using Python and Scikit-Learn to classify fraudulent and legitimate transactions.
- **Applied data preprocessing, feature engineering, and imbalance handling techniques** to improve model performance.
- **Compared multiple machine learning models** and achieved high fraud detection accuracy using **Precision, Recall, and F1-score** metrics.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)

# Set style
sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

## 1. Data Preprocessing & Exploration

In [2]:
# Load dataset
DATA_PATH = "../data/creditcard.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nClass Distribution:")
print(df['Class'].value_counts())
print(f"\nFraud Percentage: {df['Class'].mean() * 100:.3f}%")

## 2. Imbalance Handling & Feature Scaling
Since fraudulent transactions make up less than 0.2% of the dataset, we apply undersampling to construct a balanced training set and scale `Time` and `Amount` features using `StandardScaler`.

In [3]:
# Features and Target
X = df.drop('Class', axis=1)
y = df['Class']

# Train-Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Balance Training Set via Undersampling
df_train = pd.concat([X_train, y_train], axis=1)
fraud_train = df_train[df_train.Class == 1]
legit_train = df_train[df_train.Class == 0]

legit_downsampled = resample(
    legit_train,
    replace=False,
    n_samples=len(fraud_train) * 4,  # Balanced ratio
    random_state=42
)

balanced_train = pd.concat([fraud_train, legit_downsampled])
X_train_balanced = balanced_train.drop('Class', axis=1)
y_train_balanced = balanced_train['Class']

print(f"Balanced Training Set Shape: {X_train_balanced.shape}")

## 3. Machine Learning Model Comparison
We evaluate 4 different machine learning models using Scikit-Learn Pipelines:
1. **Logistic Regression**
2. **Decision Tree**
3. **Random Forest**
4. **Gradient Boosting**

In [4]:
# Define Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), ['Time', 'Amount'])
    ],
    remainder='passthrough'
)

# Define Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
fitted_pipelines = {}

for name, clf in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    pipeline.fit(X_train_balanced, y_train_balanced)
    fitted_pipelines[name] = pipeline
    
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

comparison_df = pd.DataFrame(results)
comparison_df

## 4. Visualization & Model Selection

In [5]:
# Plot comparison
plt.figure(figsize=(10, 5))
df_melted = comparison_df.melt(id_vars="Model", value_vars=["Precision", "Recall", "F1-Score", "ROC-AUC"])
sns.barplot(data=df_melted, x="Model", y="value", hue="variable", palette="viridis")
plt.title("Machine Learning Model Comparison", fontsize=14, fontweight="bold")
plt.ylim(0.7, 1.0)
plt.ylabel("Score")
plt.tight_layout()
plt.show()

## 5. Model Export
Save the best performing model pipeline (Random Forest) for production deployment in Flask.

In [6]:
# Export Winning Pipeline
best_model_name = comparison_df.sort_values(by='F1-Score', ascending=False).iloc[0]['Model']
best_pipeline = fitted_pipelines[best_model_name]

MODEL_PATH = "../model/fraud_model.pkl"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(best_pipeline, MODEL_PATH)
print(f"Best Model Pipeline ('{best_model_name}') saved successfully to {MODEL_PATH}!")